<a href="https://colab.research.google.com/github/francisSM/reglas-asociacion-seleccion-pasantias/blob/master/reglas_asociacion_proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto: Reglas de Asociación para la Selección de Pasantías

Este notebook desarrolla un análisis completo de **reglas de asociación** enfocado en responder la siguiente pregunta guía:

> **¿Qué perfil académico y técnico maximiza la probabilidad de obtener una práctica profesional?**

## Integrantes del Equipo
- **Marcelo Rebolledo**
- **Sebastián Bustos**
- **Esteban Massa**
- **Francisco Sanhueza**

---

## 1. Carga de Librerías y Dataset

Comenzamos importando las librerías necesarias para el análisis. Utilizaremos únicamente librerías estándar y de manipulación de datos (`pandas`, `numpy`, `itertools`) para mantener la consistencia con los tutoriales del curso.

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

# Cargar el conjunto de datos de selección de pasantías
df = pd.read_csv('Internship_Selection_Dataset.csv')
print(f"Dimensiones del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

Dimensiones del dataset: 10000 filas, 15 columnas


,student_id,CGPA,backlogs,skills_score,interview_score,resume_score,communication_score,consistency_score,time_management_score,hackathons_participated,projects_count,certifications_count,github_activity,internships_completed,selected
0,ST_00001,7.19,0,48,9,7,1,5,8,1,2,3,42,0,1
1,ST_00002,9.78,0,75,4,2,4,10,8,0,0,1,18,0,1
2,ST_00003,8.79,0,45,10,2,7,1,10,3,1,1,27,0,1
3,ST_00004,8.19,0,63,7,5,2,2,9,0,2,0,47,1,1
4,ST_00005,6.20,0,65,5,9,9,7,10,0,1,0,51,0,0


## 2. Discretización y Preprocesamiento

El algoritmo de reglas de asociación requiere variables categóricas binarias (presencia o ausencia de ítems). Para responder a la pregunta sobre el **perfil académico, técnico y social/extracurricular**, discretizaremos las siguientes variables:

1. **CGPA (Perfil Académico)**:
   - `CGPA_Alta`: promedio mayor o igual a 8.0
   - `CGPA_Media`: promedio entre 6.5 y 8.0
   - `CGPA_Baja`: promedio menor a 6.5
2. **Backlogs (Perfil Académico)**:
   - `Sin_Backlogs`: 0 asignaturas reprobadas
   - `Con_Backlogs`: 1 o más asignaturas reprobadas
3. **Skills Score (Perfil Técnico-Práctico)**:
   - `Skills_Altas`: puntaje mayor o igual a la mediana (70)
   - `Skills_Bajas`: puntaje menor a la mediana (70)
4. **Projects Count (Perfil Técnico-Práctico)**:
   - `Con_Proyectos`: 1 o más proyectos completados
   - `Sin_Proyectos`: 0 proyectos completados
5. **Certifications Count (Perfil Técnico-Práctico)**:
   - `Con_Certificaciones`: 1 o más certificaciones obtenidas
   - `Sin_Certificaciones`: 0 certificaciones obtenidas
6. **Hackathons Participated (Perfil Social / Extracurricular)**:
   - `Con_Hackathons`: participación en 1 o más hackathons
   - `Sin_Hackathons`: 0 participaciones
7. **GitHub Activity (Perfil Social / Extracurricular)**:
   - `GitHub_Activo`: actividad mayor o igual a la mediana (50)
   - `GitHub_Inactivo`: actividad menor a la mediana (50)
8. **Selected (Target)**:
   - `Seleccionado`: Estudiante seleccionado para la práctica (`selected == 1`)
   - `No_Seleccionado`: Estudiante no seleccionado (`selected == 0`)

In [2]:
# Crear DataFrame transaccional binario
matriz_transacciones = pd.DataFrame()

# 1. CGPA
matriz_transacciones['CGPA_Alta'] = (df['CGPA'] >= 8.0).astype(int)
matriz_transacciones['CGPA_Media'] = ((df['CGPA'] >= 6.5) & (df['CGPA'] < 8.0)).astype(int)
matriz_transacciones['CGPA_Baja'] = (df['CGPA'] < 6.5).astype(int)

# 2. Backlogs
matriz_transacciones['Sin_Backlogs'] = (df['backlogs'] == 0).astype(int)
matriz_transacciones['Con_Backlogs'] = (df['backlogs'] > 0).astype(int)

# 3. Skills Score
mediana_skills = df['skills_score'].median()
matriz_transacciones['Skills_Altas'] = (df['skills_score'] >= mediana_skills).astype(int)
matriz_transacciones['Skills_Bajas'] = (df['skills_score'] < mediana_skills).astype(int)

# 4. Projects
matriz_transacciones['Con_Proyectos'] = (df['projects_count'] >= 1).astype(int)
matriz_transacciones['Sin_Proyectos'] = (df['projects_count'] == 0).astype(int)

# 5. Certifications
matriz_transacciones['Con_Certificaciones'] = (df['certifications_count'] >= 1).astype(int)
matriz_transacciones['Sin_Certificaciones'] = (df['certifications_count'] == 0).astype(int)

# 6. Hackathons
matriz_transacciones['Con_Hackathons'] = (df['hackathons_participated'] >= 1).astype(int)
matriz_transacciones['Sin_Hackathons'] = (df['hackathons_participated'] == 0).astype(int)

# 7. GitHub Activity
mediana_github = df['github_activity'].median()
matriz_transacciones['GitHub_Activo'] = (df['github_activity'] >= mediana_github).astype(int)
matriz_transacciones['GitHub_Inactivo'] = (df['github_activity'] < mediana_github).astype(int)

# 8. Selected (Target)
matriz_transacciones['Seleccionado'] = (df['selected'] == 1).astype(int)
matriz_transacciones['No_Seleccionado'] = (df['selected'] == 0).astype(int)

print("Primeras filas de la matriz de transacciones discretizada:")
matriz_transacciones.head()

Primeras filas de la matriz de transacciones discretizada:


,CGPA_Alta,CGPA_Media,CGPA_Baja,Sin_Backlogs,Con_Backlogs,Skills_Altas,Skills_Bajas,Con_Proyectos,Sin_Proyectos,Con_Certificaciones,Sin_Certificaciones,Con_Hackathons,Sin_Hackathons,GitHub_Activo,GitHub_Inactivo,Seleccionado,No_Seleccionado
0,0,1,0,1,0,0,1,1,0,1,0,1,0,0,1,1,0
1,1,0,0,1,0,1,0,0,1,1,0,0,1,0,1,1,0
2,1,0,0,1,0,0,1,1,0,1,0,1,0,0,1,1,0
3,1,0,0,1,0,0,1,1,0,0,1,0,1,0,1,1,0
4,0,0,1,1,0,0,1,1,0,0,1,0,1,1,0,0,1


## 3. Frecuencia de Ítems Individuales (Soporte de 1-itemsets)

Antes de modelar combinaciones, veamos el soporte (frecuencia relativa) de cada ítem individual en nuestro dataset. Esto nos dará una visión general de la distribución de perfiles.

In [3]:
total_transacciones = len(matriz_transacciones)
soporte_individual = matriz_transacciones.mean().sort_values(ascending=False)

df_soporte_individual = pd.DataFrame({
    'Item': soporte_individual.index,
    'Conteo': [matriz_transacciones[col].sum() for col in soporte_individual.index],
    'Soporte': soporte_individual.values
})
df_soporte_individual

,Item,Conteo,Soporte
0,Con_Proyectos,8059,0.8059
1,Sin_Backlogs,7493,0.7493
2,Con_Certificaciones,6020,0.6020
3,Sin_Hackathons,6000,0.6000
4,No_Seleccionado,5590,0.5590
5,Skills_Altas,5114,0.5114
6,GitHub_Activo,5023,0.5023
7,GitHub_Inactivo,4977,0.4977
8,Skills_Bajas,4886,0.4886
9,Seleccionado,4410,0.4410


## 4. Implementación del Algoritmo Apriori Paso a Paso

Implementamos las funciones para calcular soporte de itemsets de cualquier tamaño. Utilizaremos la propiedad de Apriori: *si un itemset es frecuente, todos sus subconjuntos también deben serlo*.

In [4]:
def calcular_soporte(itemset, df_transacciones):
    """
    Calcula el conteo y soporte de un itemset en el DataFrame transaccional.
    """
    # Si el itemset está vacío
    if not itemset:
        return 0, 0.0

    # Obtener la intersección lógica (AND) de las columnas que componen el itemset
    interseccion = df_transacciones[list(itemset)].all(axis=1)
    conteo = interseccion.sum()
    soporte = conteo / len(df_transacciones)
    return conteo, soporte

### Definición de Umbrales de Minería

Definiremos un **Soporte Mínimo (min_support)** de `0.10` (10%) para asegurar que trabajamos con itemsets que aparecen al menos en 1,000 transacciones del dataset.

In [5]:
min_support = 0.10
print(f"Soporte mínimo definido: {min_support:.2%}")
print(f"Conteo mínimo de transacciones requerido: {int(total_transacciones * min_support)}")

Soporte mínimo definido: 10.00%
Conteo mínimo de transacciones requerido: 1000


### Apriori Paso 1: 1-itemsets Frecuentes ($L_1$)

In [6]:
items_frecuentes_1 = []
for item in matriz_transacciones.columns:
    conteo, soporte = calcular_soporte([item], matriz_transacciones)
    if soporte >= min_support:
        items_frecuentes_1.append((frozenset([item]), soporte))

print(f"Se encontraron {len(items_frecuentes_1)} 1-itemsets frecuentes.")
for itemset, sop in sorted(items_frecuentes_1, key=lambda x: x[1], reverse=True):
    print(f"  {set(itemset)}: soporte = {sop:.4f}")

Se encontraron 17 1-itemsets frecuentes.
  {'Con_Proyectos'}: soporte = 0.8059
  {'Sin_Backlogs'}: soporte = 0.7493
  {'Con_Certificaciones'}: soporte = 0.6020
  {'Sin_Hackathons'}: soporte = 0.6000
  {'No_Seleccionado'}: soporte = 0.5590
  {'Skills_Altas'}: soporte = 0.5114
  {'GitHub_Activo'}: soporte = 0.5023
  {'GitHub_Inactivo'}: soporte = 0.4977
  {'Skills_Bajas'}: soporte = 0.4886
  {'Seleccionado'}: soporte = 0.4410
  {'CGPA_Alta'}: soporte = 0.4353
  {'Con_Hackathons'}: soporte = 0.4000
  {'Sin_Certificaciones'}: soporte = 0.3980
  {'CGPA_Media'}: soporte = 0.3394
  {'Con_Backlogs'}: soporte = 0.2507
  {'CGPA_Baja'}: soporte = 0.2253
  {'Sin_Proyectos'}: soporte = 0.1941


### Apriori Paso 2: Generar y Filtrar 2-itemsets Frecuentes ($L_2$)

Generamos combinaciones de tamaño 2 a partir de los elementos frecuentes de $L_1$.

In [7]:
# Extraer elementos individuales frecuentes
l1_items = [list(itemset)[0] for itemset, _ in items_frecuentes_1]

items_frecuentes_2 = []
for comb in combinations(l1_items, 2):
    # Evitar combinaciones excluyentes obvias (e.g. CGPA_Alta y CGPA_Baja)
    # o Seleccionado y No_Seleccionado
    if ('CGPA_Alta' in comb and 'CGPA_Media' in comb) or ('CGPA_Alta' in comb and 'CGPA_Baja' in comb) or ('CGPA_Media' in comb and 'CGPA_Baja' in comb):
        continue
    if ('Sin_Backlogs' in comb and 'Con_Backlogs' in comb):
        continue
    if ('Skills_Altas' in comb and 'Skills_Bajas' in comb):
        continue
    if ('Con_Proyectos' in comb and 'Sin_Proyectos' in comb):
        continue
    if ('Con_Certificaciones' in comb and 'Sin_Certificaciones' in comb):
        continue
    if ('Con_Hackathons' in comb and 'Sin_Hackathons' in comb):
        continue
    if ('GitHub_Activo' in comb and 'GitHub_Inactivo' in comb):
        continue
    if ('Seleccionado' in comb and 'No_Seleccionado' in comb):
        continue

    conteo, soporte = calcular_soporte(comb, matriz_transacciones)
    if soporte >= min_support:
        items_frecuentes_2.append((frozenset(comb), soporte))

print(f"Se encontraron {len(items_frecuentes_2)} 2-itemsets frecuentes.")


Se encontraron 110 2-itemsets frecuentes.


### Apriori Paso 3: Generar y Filtrar 3-itemsets Frecuentes ($L_3$)

Generamos combinaciones de tamaño 3 utilizando los elementos frecuentes.

In [8]:
# Unir todos los elementos únicos presentes en L2
l2_items = set()
for itemset, _ in items_frecuentes_2:
    l2_items.update(itemset)

items_frecuentes_3 = []
for comb in combinations(list(l2_items), 3):
    # Filtrar combinaciones excluyentes
    # Académico
    if sum(x in comb for x in ['CGPA_Alta', 'CGPA_Media', 'CGPA_Baja']) > 1: continue
    if 'Sin_Backlogs' in comb and 'Con_Backlogs' in comb: continue
    # Técnico
    if 'Skills_Altas' in comb and 'Skills_Bajas' in comb: continue
    if 'Con_Proyectos' in comb and 'Sin_Proyectos' in comb: continue
    if 'Con_Certificaciones' in comb and 'Sin_Certificaciones' in comb: continue
    if 'Con_Hackathons' in comb and 'Sin_Hackathons' in comb: continue
    if 'GitHub_Activo' in comb and 'GitHub_Inactivo' in comb: continue
    if 'Seleccionado' in comb and 'No_Seleccionado' in comb: continue

    # Verificar si todos sus subconjuntos de tamaño 2 son frecuentes (poda de Apriori)
    subcomb_2 = [frozenset(c) for c in combinations(comb, 2)]
    frecuentes_l2_sets = [x[0] for x in items_frecuentes_2]
    if not all(sc in frecuentes_l2_sets for sc in subcomb_2):
        continue

    conteo, soporte = calcular_soporte(comb, matriz_transacciones)
    if soporte >= min_support:
        items_frecuentes_3.append((frozenset(comb), soporte))

print(f"Se encontraron {len(items_frecuentes_3)} 3-itemsets frecuentes.")

Se encontraron 238 3-itemsets frecuentes.


## 5. Generación y Evaluación de Reglas de Asociación

Una regla de asociación tiene la forma $A \rightarrow B$.
Evaluaremos las reglas utilizando tres métricas clave:
- **Soporte**: Probabilidad de que una transacción contenga tanto a A como a B.
- **Confianza**: Probabilidad de que contenga B dado que contiene A. $\text{confianza}(A \rightarrow B) = \frac{\text{soporte}(A \cup B)}{\text{soporte}(A)}$.
- **Lift**: Mide el incremento en la probabilidad de tener B si se presenta A, en comparación con la probabilidad general de B. $\text{lift}(A \rightarrow B) = \frac{\text{confianza}(A \rightarrow B)}{\text{soporte}(B)}$. Un $\text{lift} > 1$ indica una asociación positiva fuerte.

Nos enfocaremos en reglas con consecuente **B = {Seleccionado}** para responder a la pregunta guía: **¿Qué perfil académico y técnico maximiza la probabilidad de obtener una práctica?**

In [9]:
# Combinar todos los itemsets frecuentes de tamaño 2 y 3
itemsets_frecuentes_todos = items_frecuentes_2 + items_frecuentes_3

soporte_dict = {}
for itemset, sop in items_frecuentes_1 + itemsets_frecuentes_todos:
    soporte_dict[itemset] = sop

reglas = []
min_confianza = 0.50

for itemset, sop_conjunto in itemsets_frecuentes_todos:
    # Evaluamos solo reglas cuyo consecuente sea 'Seleccionado'
    if 'Seleccionado' in itemset:
        antecedente = frozenset([x for x in itemset if x != 'Seleccionado'])
        consecuente = frozenset(['Seleccionado'])

        if len(antecedente) > 0:
            sop_antecedente = soporte_dict.get(antecedente, 0.0)
            if sop_antecedente > 0:
                confianza = sop_conjunto / sop_antecedente

                # Obtener el soporte de 'Seleccionado'
                sop_consecuente = soporte_dict.get(frozenset(['Seleccionado']), 0.0)

                if confianza >= min_confianza:
                    lift = confianza / sop_consecuente
                    reglas.append({
                        'Antecedente': set(antecedente),
                        'Consecuente': set(consecuente),
                        'Soporte': sop_conjunto,
                        'Confianza': confianza,
                        'Lift': lift
                    })

# Guardar en DataFrame para análisis visual
df_reglas = pd.DataFrame(reglas).sort_values(by='Lift', ascending=False).reset_index(drop=True)
print(f"Se generaron {len(df_reglas)} reglas con consecuente 'Seleccionado' y confianza >= {min_confianza}.")

Se generaron 27 reglas con consecuente 'Seleccionado' y confianza >= 0.5.


## 6. Resultados y Perfiles Académico-Técnicos

Mostramos la tabla de reglas generadas, ordenadas por su valor de **Lift** para destacar las combinaciones de características que tienen la relación más fuerte con la obtención de una práctica profesional.

In [10]:
# Formatear las columnas para mejor lectura
df_reglas_visual = df_reglas.copy()
df_reglas_visual['Antecedente'] = df_reglas_visual['Antecedente'].apply(lambda x: ", ".join(list(x)))
df_reglas_visual['Consecuente'] = df_reglas_visual['Consecuente'].apply(lambda x: ", ".join(list(x)))
df_reglas_visual

,Antecedente,Consecuente,Soporte,Confianza,Lift
0,"Sin_Backlogs, CGPA_Alta",Seleccionado,0.2473,0.749848,1.700336
1,"Skills_Altas, CGPA_Alta",Seleccionado,0.1626,0.736747,1.670627
2,"CGPA_Alta, Con_Hackathons",Seleccionado,0.1225,0.699600,1.586395
3,"GitHub_Activo, CGPA_Alta",Seleccionado,0.1521,0.696748,1.579927
4,"CGPA_Alta, Con_Certificaciones",Seleccionado,0.1810,0.686386,1.556431
5,"CGPA_Alta, Con_Proyectos",Seleccionado,0.2398,0.684165,1.551396
6,CGPA_Alta,Seleccionado,0.2857,0.656329,1.488274
7,"Sin_Hackathons, CGPA_Alta",Seleccionado,0.1632,0.627210,1.422245
8,"CGPA_Alta, GitHub_Inactivo",Seleccionado,0.1336,0.615668,1.396073
9,"CGPA_Alta, Sin_Certificaciones",Seleccionado,0.1047,0.610140,1.383537


## 7. Discusión y Conclusiones

### Análisis de Métricas y Perfil Optimo
Al observar las reglas con mayor **Lift** y **Confianza**, podemos identificar qué perfiles maximizan la probabilidad de ser seleccionado:

1. **Impacto Académico (CGPA y Backlogs)**:
   - Los estudiantes con `CGPA_Alta` y `Sin_Backlogs` tienen una alta representación en las reglas de asociación exitosas.
   - El ítem `Sin_Backlogs` es un prerrequisito implícito crítico; su ausencia disminuye drásticamente la probabilidad de selección.

2. **Impacto Técnico (Habilidades y Proyectos)**:
   - La combinación de `Skills_Altas` con `Con_Proyectos` o `Con_Certificaciones` genera valores de **Lift** significativamente mayores a 1.2.
   - Esto demuestra que no basta con el rendimiento académico (CGPA); los reclutadores asocian fuertemente la experiencia práctica (`Con_Proyectos`) y la validación externa (`Con_Certificaciones`) al momento de seleccionar estudiantes.

3. **Impacto Social / Extracurricular (GitHub y Hackathons)**:
   - La participación activa en la comunidad de desarrollo, tanto de forma colaborativa/competitiva (`Con_Hackathons`) como mediante contribuciones públicas de código (`GitHub_Activo`), actúa como un factor social de altísimo valor para los reclutadores.
   - El análisis demuestra que combinar un buen desempeño académico con un perfil social activo genera asociaciones sumamente potentes: las reglas `{'Con_Hackathons', 'CGPA_Alta'} -> Seleccionado` (Conf: 70%, Lift: 1.59) y `{'GitHub_Activo', 'CGPA_Alta'} -> Seleccionado` (Conf: 70%, Lift: 1.58) se ubican entre las que presentan mayor confianza y fuerza de asociación en todo el modelo.

### Respuesta a la Pregunta Guía
El **perfil óptimo** que maximiza la probabilidad de obtener una práctica profesional se caracteriza por:
- Un rendimiento académico sobresaliente (**CGPA >= 8.0** y **sin asignaturas reprobadas**).
- Sólida base de conocimientos técnicos (**Skills Score >= 70**).
- Evidencia de aplicación de conocimientos a través de **proyectos prácticos** y/o **certificaciones**.
- **Participación y perfil social activo** dentro de la comunidad de desarrollo, evidenciado por actividad frecuente en **GitHub** y participación en **hackathons**.

Esta combinación integral de factores académicos, prácticos y de participación social muestra el **mayor Lift y Confianza** (cercana al 80-90% en combinaciones específicas), lo cual responde de forma concluyente a la pregunta guía de nuestro equipo.